# PennyLane VQE

Minimize a two-qubit Hamiltonian with parameter-shift gradients and compare the complete energy trace.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

VQE minimizes a Hamiltonian expectation using a parameterized quantum state and a classical optimizer.

In [2]:
hamiltonian = -1.05 * qml.I(0) + 0.39 * qml.Z(0) - 0.39 * qml.Z(1) - 0.01 * (qml.Z(0) @ qml.Z(1)) + 0.18 * (qml.X(0) @ qml.X(1))

def make_energy(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def energy(weights):
        qml.RY(weights[0], wires=0)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[1], wires=1)
        return qml.expval(hamiltonian)
    return energy

def optimize(energy):
    weights = pnp.array([0.2, -0.3], requires_grad=True)
    trace = []
    for _ in range(10):
        trace.append(float(energy(weights)))
        weights = weights - 0.12 * qml.grad(energy)(weights)
    trace.append(float(energy(weights)))
    return np.asarray(trace)

reference_energy = make_energy(qml.device("default.qubit", wires=2))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: optimize(reference_energy), repeats=2)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_energy = make_energy(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: optimize(mettleq_energy), repeats=2)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

Every energy in the optimization trace must match within tolerance.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/05_vqe.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="VQE energy trace atol=5e-5",
    passed=error <= 5e-5 and candidate[-1] < candidate[0],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_energy_error": error, "reference_final": reference[-1], "mettleq_final": candidate[-1]},
)


Comparison summary
------------------
Correctness contract: PASS — VQE energy trace atol=5e-5
SDK reference median: 64.782 ms
MettleQ median:       321.869 ms
Timing interpretation: the SDK reference was 4.968x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "VQE energy trace atol=5e-5", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_energy_error": 1.3775572171859096e-07, "mettleq_final": -1.0538829098641873, "reference_final": -1.0538829210415113}, "mettleq_median_ms": 321.8692084919894, "notebook": "pennylane/05_vqe.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 64.78208350017667, "reference_over_mettleq": 0.2012683468657703, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

The backend becomes relevant as the ansatz widens; this minimal molecule-style example emphasizes workflow compatibility.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.